In [1]:
import numpy as np
import scipy
import sympy
import unittest

In [2]:
# Derivation of the two-population genotype estimator after Ragsdale and Gravel 2020

c11, c12, c13, c13, g11, g12, g13, g14, g15, g16, g17, g18, g19 = \
    sympy.symbols("c11 c12 c13 c14 g11 g12 g13 g14 g15 g16 g17 g18 g19")
c21, c22, c23, c24, g21, g22, g23, g24, g25, g26, g27, g28, g29 = \
    sympy.symbols("c21 c22 c23 c24 g21 g22 g23 g24 g25 g26 g27 g28 g29")

c11 = g11 + g12 / 2 + g14 / 2 + g15 / 4
c12 = g12 / 2 + g13 + g15 / 4 + g16 / 2
c13 = g14 / 2 + g15 / 4 + g17 + g18 / 2
c14 = g15 / 4 + g16 / 2 + g18 / 2 + g19

c21 = g21 + g22 / 2 + g24 / 2 + g25 / 4
c22 = g22 / 2 + g23 + g25 / 4 + g26 / 2
c23 = g24 / 2 + g25 / 4 + g27 + g28 / 2
c24 = g25 / 4 + g26 / 2 + g28 / 2 + g29

expr = str(sympy.expand(c11 * c24 + c21 * c14 + c12 * c23 + c22 * c13))
# print(expr.replace(" + ", "\n+ ").replace("*", " * ").replace("/", " / "))

In [ ]:
def tally_haplotype_pairs(haplotypes, idx_i=None, idx_j=None, sample_indices=None):
    """
    Compute two-locus haplotype counts for given locus pairs.
    
    Parameters
    ----------
    haplotypes : np.ndarray
        Shape (n_loci, n_haplotypes). Values 0, 1.

    idx_i, idx_j : list
        Length n_pairs. Lists specifying locus pairs. When not given,
        all locus pairs are included.

    sample_indices : list
        Indices of haplotypes to include.
    """
    if sample_indices is None:
        sample_indices = list(range(haplotypes.shape[1]))
    haplotypes = haplotypes[:, sample_indices]

    if idx_i is None and idx_j is None:
        n_loci = genotypes.shape[0]
        idx_i = [i for i in range(n_loci) for j in range(i + 1, n_loci)]
        idx_j = [j for i in range(n_loci) for j in range(i + 1, n_loci)]
    else:
        assert idx_i is not None and idx_j is not None
    
    locus_i = haplotypes[idx_i]
    locus_j = haplotypes[idx_j]

    n11 = np.sum((locus_i == 1) & (locus_j == 1), axis=1)
    n10 = np.sum((locus_i == 1) & (locus_j == 0), axis=1)
    n01 = np.sum((locus_i == 0) & (locus_j == 1), axis=1)
    n00 = np.sum((locus_i == 0) & (locus_j == 0), axis=1)

    counts = np.stack([n11, n10, n01, n00], axis=1)
    return counts

In [18]:
def tally_genotype_pairs(genotypes, idx_i=None, idx_j=None, sample_indices=None):
    """
    Compute two-locus genotype counts for given locus pairs.

    Not very efficient.
    
    Parameters
    ----------
    genotypes : np.ndarray
        Shape (n_loci, n_samples). Values 0, 1, 2.

    idx_i, idx_j : list
        Length n_pairs. Lists specifying locus pairs. When not given,
        all locus pairs are included.

    sample_indices : list
        Indices of genotypes to include.
    """
    if sample_indices is None:
        sample_indices = list(range(genotypes.shape[1]))
    genotypes = genotypes[:, sample_indices]

    if idx_i is None and idx_j is None:
        n_loci = genotypes.shape[0]
        idx_i = [i for i in range(n_loci) for j in range(i + 1, n_loci)]
        idx_j = [j for i in range(n_loci) for j in range(i + 1, n_loci)]
    else:
        assert idx_i is not None and idx_j is not None
    
    locus_i = genotypes[idx_i]
    locus_j = genotypes[idx_j]

    n22 = np.sum((locus_i == 2) & (locus_j == 2), axis=1)
    n21 = np.sum((locus_i == 2) & (locus_j == 1), axis=1)
    n20 = np.sum((locus_i == 2) & (locus_j == 0), axis=1)
    n12 = np.sum((locus_i == 1) & (locus_j == 2), axis=1)
    n11 = np.sum((locus_i == 1) & (locus_j == 1), axis=1)
    n10 = np.sum((locus_i == 1) & (locus_j == 0), axis=1)
    n02 = np.sum((locus_i == 0) & (locus_j == 2), axis=1)
    n01 = np.sum((locus_i == 0) & (locus_j == 1), axis=1)
    n00 = np.sum((locus_i == 0) & (locus_j == 0), axis=1)

    counts = np.stack([n22, n21, n20, n12, n11, n10, n02, n01, n00], axis=1)
    return counts

In [3]:
def h2_haplotype_within(counts, pop_idx):
    """Calculate within-population H2 from two-locus haplotype counts"""
    start = 4 * pop_idx
    c1, c2, c3, c4 = counts[:, start:start + 4].T
    n = np.sum(counts[start:start + 4], axis=1)
    numer = c1 * c4 + c2 * c3
    denom = n * (n - 1) / 2
    stat = numer / denom
    return stat


def h2_haplotype_within_enum(counts, pop_idx):
    """Calculate H2 by making all possible pairwise haplotype comparisons"""
    start = 4 * pop_idx
    pop_counts = counts[:, start:start + 4]
    sample = [i + 1 for i, n in enumerate(pop_counts) for _ in range(n)]
    numer = 0.0
    denom = 0
    for i, hap_i in enumerate(sample):
        for hap_j in sample[i + 1:]:
            hap_1, hap_2 = sorted([hap_i, hap_j])
            if hap_1 == 1 and hap_2 == 4:
                numer += 1.0
            elif hap_1 == 2 and hap_2 == 3:
                numer += 1.0
            denom += 1
    stat = numer / denom
    return stat


class TestH2HaplotypeWithin(unittest.TestCase):

    def test_diploid_samples(self):
        sample, exp_stat = np.array([[1, 0, 0, 1]]), 1.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 1, 1, 0]]), 1.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[2, 0, 0, 0]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 2, 0, 0]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 0, 2, 0]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 0, 0, 2]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 1, 0, 1]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
    def test_two_diploid_samples(self):
        sample, exp_stat = np.array([[2, 0, 0, 2]]), 2 / 3
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 2, 2, 0]]), 2 / 3
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[1, 0, 0, 3]]), 0.5
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[3, 0, 0, 1]]), 0.5
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 3, 1, 0]]), 0.5
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 1, 3, 0]]), 0.5
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[4, 0, 0, 0]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)
        
        sample, exp_stat = np.array([[0, 4, 0, 0]]), 0.0
        self.assertEqual(h2_haplotype_within(sample, 0), exp_stat)

    def test_multiple_sites(self):
        sample = np.array([[1, 0, 0, 1], [2, 0, 0, 0]])
        exp_stat = np.array([1, 0])
        self.assertTrue(np.all(h2_haplotype_within(sample, 0) == exp_stat))

In [4]:
def h2_haplotype_between(counts, pop1_idx, pop2_idx):
    """Calculate between-population H2 from two-locus haplotype counts"""
    start1 = pop1_idx * 4
    start2 = pop2_idx * 4
    c11, c12, c13, c14 = counts[:, start1:start1 + 4].T
    c21, c22, c23, c24 = counts[:, start2:start2 + 4].T
    n1 = np.sum(counts[:, start1:start1 + 4])
    n2 = np.sum(counts[:, start2:start2 + 4])
    numer = c11 * c24 + c21 * c14 + c12 * c23 + c22 * c13
    denom = n1 * n2
    stat = numer / denom
    return stat


class TestH2HaplotypeBetween(unittest.TestCase):

    def test_haploid_samples(self):
        counts, exp_stat = np.array([[1, 0, 0, 0, 0, 0, 0, 1]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 1, 1, 0, 0, 0]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 1, 0, 0, 1, 0, 0]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[0, 1, 0, 0, 0, 0, 1, 0]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)

        counts, exp_stat = np.array([[0, 0, 0, 1, 0, 0, 0, 1]]), 0.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)

        counts, exp_stat = np.array([[0, 1, 0, 0, 0, 1, 0, 0]]), 0.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)

    def test_diploid_samples(self):
        counts, exp_stat = np.array([[2, 0, 0, 0, 0, 0, 0, 2]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[0, 2, 0, 0, 0, 0, 2, 0]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 2, 0, 0, 2, 0, 0]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 2, 2, 0, 0, 0]]), 1.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[1, 0, 0, 1, 0, 0, 0, 2]]), 0.5
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
        
        counts, exp_stat = np.array([[2, 0, 0, 0, 1, 0, 0, 1]]), 0.5
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
                        
        counts, exp_stat = np.array([[0, 0, 2, 0, 0, 1, 1, 0]]), 0.5
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
                        
        counts, exp_stat = np.array([[0, 0, 2, 0, 0, 1, 0, 1]]), 0.5
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
                        
        counts, exp_stat = np.array([[0, 1, 1, 0, 0, 1, 1, 0]]), 0.5
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
                        
        counts, exp_stat = np.array([[1, 0, 1, 0, 0, 1, 1, 0]]), 0.25
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
                        
        counts, exp_stat = np.array([[1, 0, 1, 0, 0, 0, 1, 1]]), 0.25
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)
                        
        counts, exp_stat = np.array([[1, 0, 1, 0, 0, 0, 2, 0]]), 0.0
        self.assertEqual(h2_haplotype_between(counts, 0, 1), exp_stat)

In [5]:
def h2_genotype_within(counts, pop_idx):
    """Compute within-population H2 from genotype counts"""
    start = 9 * pop_idx
    g1, g2, g3, g4, g5, g6, g7, g8, g9 = counts[:, start:start + 9].T
    n = np.sum(counts[:, start:start + 9], axis=1)
    numer = (
        g1 * g5
        + 2 * g1 * g6
        + 2 * g1 * g8
        + 4 * g1 * g9
        + g2 * g4
        + g2 * g5
        + g2 * g6
        + 2 * g2 * g7
        + 2 * g2 * g8
        + 2 * g2 * g9
        + 2 * g3 * g4
        + g3 * g5
        + 4 * g3 * g7
        + 2 * g3 * g8
        + g4 * g5
        + 2 * g4 * g6
        + g4 * g8
        + 2 * g4 * g9
        + g5 * (g5 + 1) / 2
        + g5 * g6
        + g5 * g7
        + g5 * g8
        + g5 * g9
        + 2 * g6 * g7
        + g6 * g8
        )
    denom = n * (2 * n - 1)
    stat = numer / denom
    return stat


def get_possible_haplotype_configs(counts):
    """
    Find the possible haplotype configurations, and their relative weights given the
    assumption of linkage equilibrium, from genotype counts.
    """
    g1, g2, g3, g4, g5, g6, g7, g8, g9 = counts
    # Calculate the numbers of haplotypes from non-doubly-heterzygous genotypes
    known = np.sum([
        g1 * np.array([[2, 0, 0, 0]]),
        g2 * np.array([[1, 1, 0, 0]]),
        g3 * np.array([[0, 2, 0, 0]]),
        g4 * np.array([[1, 0, 1, 0]]),
        g6 * np.array([[0, 1, 0, 1]]),
        g7 * np.array([[0, 0, 2, 0]]),
        g8 * np.array([[0, 0, 1, 1]]),
        g9 * np.array([[0, 0, 0, 2]]),
        ], axis=0)
    # The haplotypes that underly double heterzygote genotypes are unknown. We take the
    # probabilities of coupling/repulsion phasings to be equal. The number of coupling
    # haplotype pairs from double heterozygotes is then Binomial p=1/2, n=g5.
    weights = []
    configs = []
    for n_coupling in range(g5 + 1):
        weight = scipy.special.binom(g5, n_coupling) * 0.5 ** g5
        weights.append(weight)
        types = np.sum([
        n_coupling * np.array([[1, 0, 0, 1]]),
        (g5 - n_coupling) * np.array([[0, 1, 1, 0]]),
        ], axis=0)
        configs.append(known + types)
    return weights, configs


def h2_genotype_within_validator(counts, pop_idx, site=0):
    """Calculate H2 from genotypes by averaging across possible haplotype configs"""
    start = 9 * pop_idx
    pop_counts = counts[site, start:start + 9]
    weights, configs = get_possible_haplotype_configs(pop_counts)
    stat = np.sum([w * h2_haplotype_within(config, 0) 
                   for w, config in zip(weights, configs)])
    return stat


class TestH2GenotypeWithin(unittest.TestCase):

    def test_diploid_samples(self):
        counts, exp_stat = np.array([[1, 0, 0, 0, 0, 0, 0, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 1, 0, 0, 0, 0, 0, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 1, 0, 0, 0, 0, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 1, 0, 0, 0, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 1, 0, 0, 0, 0]]), 1.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 0, 1, 0, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 0, 0, 1, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 0, 0, 0, 1, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 1]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)

    def test_two_diploid_samples(self):
        counts, exp_stat = np.array([[2, 0, 0, 0, 0, 0, 0, 0, 0]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 0, 0, 0, 0, 2]]), 0.0
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[1, 0, 0, 0, 0, 0, 0, 0, 1]]), 2 / 3
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 2, 0, 0, 0, 0]]), 0.5
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 1, 0, 0, 0, 0, 0, 0, 1]]), 1 / 3
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[0, 0, 0, 0, 1, 0, 0, 0, 1]]), 1 / 3
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
        counts, exp_stat = np.array([[1, 0, 0, 0, 1, 0, 0, 0, 0]]), 1 / 3
        self.assertEqual(h2_genotype_within(counts, 0), exp_stat)
        
    def test_large_samples(self):
        counts = np.array([[5, 1, 2, 3, 5, 0, 2, 2, 5]])
        self.assertEqual(h2_genotype_within_validator(counts, 0), 
                         h2_genotype_within(counts, 0))
        
        counts = np.array([[6, 6, 6, 6, 6, 6, 6, 6, 6]])
        self.assertEqual(h2_genotype_within_validator(counts, 0), 
                         h2_genotype_within(counts, 0))
        
        counts = np.array([[0, 0, 0, 0, 30, 0, 0, 0, 0]])
        self.assertTrue(np.isclose(
            h2_genotype_within_validator(counts, 0), 
            h2_genotype_within(counts, 0)))

In [6]:
def _h2_genotype_between(counts, pop1_idx, pop2_idx):
    """Compute between-population H2 from genotype counts (slower version)"""
    start1 = 9 * pop1_idx
    start2 = 9 * pop2_idx
    g1, g2, g3, g4, g5, g6, g7, g8, g9 = counts[:, start1:start1 + 9].T
    g1, g2, g3, g4, g5, g6, g7, g8, g9 = counts[:, start2:start2 + 9].T
    n1 = np.sum(counts[:, start1:start1 + 9], axis=1)
    n2 = np.sum(counts[:, start2:start2 + 9], axis=1)
    numer = (
        g11 * g25 / 4
        + g11 * g26 / 2
        + g11 * g28 / 2
        + g11 * g29
        + g12 * g24 / 4
        + g12 * g25 / 4
        + g12 * g26 / 4
        + g12 * g27 / 2
        + g12 * g28 / 2
        + g12 * g29 / 2
        + g13 * g24 / 2
        + g13 * g25 / 4
        + g13 * g27
        + g13 * g28 / 2
        + g14 * g22 / 4
        + g14 * g23 / 2
        + g14 * g25 / 4
        + g14 * g26 / 2
        + g14 * g28 / 4
        + g14 * g29 / 2
        + g15 * g21 / 4
        + g15 * g22 / 4
        + g15 * g23 / 4
        + g15 * g24 / 4
        + g15 * g25 / 4
        + g15 * g26 / 4
        + g15 * g27 / 4
        + g15 * g28 / 4
        + g15 * g29 / 4
        + g16 * g21 / 2
        + g16 * g22 / 4
        + g16 * g24 / 2
        + g16 * g25 / 4
        + g16 * g27 / 2
        + g16 * g28 / 4
        + g17 * g22 / 2
        + g17 * g23
        + g17 * g25 / 4
        + g17 * g26 / 2
        + g18 * g21 / 2
        + g18 * g22 / 2
        + g18 * g23 / 2
        + g18 * g24 / 4
        + g18 * g25 / 4
        + g18 * g26 / 4
        + g19 * g21
        + g19 * g22 / 2
        + g19 * g24 / 2
        + g19 * g25 / 4
        )
    denom = n1 * n2
    stat = numer / denom
    return stat


def h2_genotype_between(counts, pop1_idx, pop2_idx):
    """Compute between-sample H2 from genotype counts"""
    start1 = 9 * pop1_idx
    start2 = 9 * pop2_idx
    g11, g12, g13, g14, g15, g16, g17, g18, g19 = counts[:, start1:start1 + 9].T
    g21, g22, g23, g24, g25, g26, g27, g28, g29 = counts[:, start2:start2 + 9].T
    n1 = np.sum(counts[:, start1:start1 + 9], axis=1)
    n2 = np.sum(counts[:, start2:start2 + 9], axis=1)
    numer = (
        (g11 + g12 / 2 + g14 / 2 + g15 / 4) 
        * (g25 / 4 + g26 / 2 + g28 / 2 + g29)
        + (g15 / 4 + g16 / 2 + g18 / 2 + g19) 
        * (g21 + g22 / 2 + g24 / 2 + g25 / 4)
        + (g12 / 2 + g13 + g15 / 4 + g16 / 2) 
        * (g24 / 2 + g25 / 4 + g27 + g28 / 2)
        + (g14 / 2 + g15 / 4 + g17 + g18 / 2) 
        * (g22 / 2 + g23 + g25 / 4 + g26 / 2)
        )
    denom = n1 * n2
    stat = numer / denom
    return stat

    
def h2_genotype_between_validator(counts, pop1_idx, pop2_idx, site=0):
    """Calculate between-poplation H2 from genotypes by averaging over haplotype configs"""
    start1 = 9 * pop1_idx
    pop1_counts = counts[site, start1:start1 + 9]
    start2 = 9 * pop2_idx
    pop2_counts = counts[site, start2:start2 + 9]
    weights1, configs1 = get_possible_haplotype_configs(pop1_counts)
    weights2, configs2 = get_possible_haplotype_configs(pop2_counts)
    stat = 0.0
    for weight1, config1 in zip(weights1, configs1):
        for weight2, config2 in zip(weights2, configs2):
            pr = weight1 * weight2
            cfg_counts = np.concatenate([config1, config2], axis=1)
            stat += pr * h2_haplotype_between(cfg_counts, 0, 1)
    return stat


class TestH2GenotypeBetween(unittest.TestCase):

    def test_diploid_samples(self):
        counts = np.array([[1, 0, 0, 0, 0, 0, 0, 0, 0,
                            0, 0, 0, 0, 0, 0, 0, 0, 1]])
        exp_stat = 1.0
        self.assertEqual(h2_genotype_between(counts, 0, 1), exp_stat)
        
        counts = np.array([[1, 0, 0, 0, 0, 0, 0, 0, 0,
                            0, 0, 0, 0, 1, 0, 0, 0, 0]])
        exp_stat = 0.25
        self.assertEqual(h2_genotype_between(counts, 0, 1), exp_stat)
        
        counts = np.array([[0, 0, 0, 0, 1, 0, 0, 0, 0,
                            0, 0, 0, 0, 0, 0, 0, 0, 1]])
        exp_stat = 0.25
        self.assertEqual(h2_genotype_between(counts, 0, 1), exp_stat)
    
    def test_large_samples(self):
        counts = np.array([[5, 1, 1, 2, 5, 2, 0, 0, 2,
                            3, 1, 2, 1, 5, 1, 0, 2, 4]])
        self.assertEqual(h2_genotype_between_validator(counts, 0, 1),
                         h2_genotype_between(counts, 0, 1))
        
        counts = np.array([[0, 0, 0, 0, 20, 0, 0, 0, 0,
                            0, 0, 0, 0, 20, 0, 0, 0, 0]])
        self.assertTrue(np.isclose(
            h2_genotype_between_validator(counts, 0, 1), 
            h2_genotype_between(counts, 0, 1)))

In [26]:
h2_genotype_within(np.array([[0, 0, 0, 0.01, 1.89, 0.1, 0, 0, 0]]), 0)

array([0.49015833])

In [53]:
def compute_expected_two_locus_genotypes(gprobs, idx_i=None, idx_j=None, sample_indices=None):
    """
    Compute expected tallies of two-locus genotypes from genotype probabilities.

    Parameters
    ----------
    gprobs : np.ndarray
        Shape (n_loci, 3 * n_samples). For sample i, columns i, i + 1, i + 2 hold
        the posterior probabilities assigned to genotypes 0/0, 0/1, 1/1.
    """
    if sample_indices is None:
        sample_indices = list(range(gprobs.shape[1]))
    gprobs = gprobs[:, sample_indices]

    if idx_i is None and idx_j is None:
        n_loci = gprobs.shape[0]
        idx_i = [i for i in range(n_loci) for j in range(i + 1, n_loci)]
        idx_j = [j for i in range(n_loci) for j in range(i + 1, n_loci)]
    else:
        assert idx_i is not None and idx_j is not None
    
    locus_i = gprobs[idx_i]
    locus_j = gprobs[idx_j]

    n22 = np.sum(locus_i[:, 2::3] * locus_j[:, 2::3], axis=1)
    n21 = np.sum(locus_i[:, 2::3] * locus_j[:, 1::3], axis=1)
    n20 = np.sum(locus_i[:, 2::3] * locus_j[:, ::3], axis=1)
    n12 = np.sum(locus_i[:, 1::3] * locus_j[:, 2::3], axis=1)
    n11 = np.sum(locus_i[:, 1::3] * locus_j[:, 1::3], axis=1)
    n10 = np.sum(locus_i[:, 1::3] * locus_j[:, ::3], axis=1)
    n02 = np.sum(locus_i[:, ::3] * locus_j[:, 2::3], axis=1)
    n01 = np.sum(locus_i[:, ::3] * locus_j[:, 1::3], axis=1)
    n00 = np.sum(locus_i[:, ::3] * locus_j[:, ::3], axis=1)
    
    exp_counts = np.stack([n22, n21, n20, n12, n11, n10, n02, n01, n00], axis=1)
    return exp_counts


gprobs = np.array([[0., 0.99, 0.01,], [0.01, 0.99, 0.,]])
print(compute_expected_two_locus_genotypes(gprobs))
h2_genotype_within(compute_expected_two_locus_genotypes(gprobs), 0)

[[0.000e+00 9.900e-03 1.000e-04 0.000e+00 9.801e-01 9.900e-03 0.000e+00
  0.000e+00 0.000e+00]]


array([0.98995])

In [43]:
unittest.main(argv=[""], verbosity=2, exit=False) 

test_diploid_samples (__main__.TestH2GenotypeBetween.test_diploid_samples) ... ok
test_large_samples (__main__.TestH2GenotypeBetween.test_large_samples) ... ok
test_diploid_samples (__main__.TestH2GenotypeWithin.test_diploid_samples) ... ok
test_large_samples (__main__.TestH2GenotypeWithin.test_large_samples) ... ok
test_two_diploid_samples (__main__.TestH2GenotypeWithin.test_two_diploid_samples) ... ok
test_diploid_samples (__main__.TestH2HaplotypeBetween.test_diploid_samples) ... ok
test_haploid_samples (__main__.TestH2HaplotypeBetween.test_haploid_samples) ... ok
test_diploid_samples (__main__.TestH2HaplotypeWithin.test_diploid_samples) ... ok
test_multiple_sites (__main__.TestH2HaplotypeWithin.test_multiple_sites) ... ok
test_two_diploid_samples (__main__.TestH2HaplotypeWithin.test_two_diploid_samples) ... ok

----------------------------------------------------------------------
Ran 10 tests in 0.031s

OK
